# Phase 5: Imputation of Missing Values (MCAR and MNAR)

This notebook generates missing-value imputation results using the trained MIMIR
shared latent space model.

We consider two missingness mechanisms:
1. **MCAR (Missing Completely At Random)**, where values are randomly masked
2. **MNAR (Missing Not At Random)**, where low-valued entries are preferentially masked

This notebook **only generates masks, corrupted inputs, and MIMIR predictions**.
All benchmarking and comparison against baselines (SoftImpute, KNN) is performed
in the subsequent notebook.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import random
from typing import Dict, List, Tuple
import pickle
from torch.optim import Adam
import matplotlib.pyplot as plt
import pandas as pd
import json

In [2]:
from src.data_utils import *
from src.mae_masked import *
from src.shared_finetune import *
from src.impute1 import *
from src.translation import *
from src.evaluation import *

## 0.1 Loading data and trained model

We load the z-scored multi-omic dataset and reuse the same train/validation/test
splits used throughout the pipeline.

The pretrained modality-specific autoencoders (Phase 1) and the shared latent
space model (Phase 2) are reconstructed and loaded from disk.


In [3]:
with open('tcga_redo_mlomicZ.pkl', 'rb') as f:
    multi_omic_data = pickle.load(f)
    
print(multi_omic_data.keys())

common_samples, train_idx, val_idx, test_idx = load_shared_splits_from_json(
    multi_omic_data,
    json_path='splits.json'
)
print(f"Shared N={len(common_samples)} | train={len(train_idx)} | val={len(val_idx)}| test={len(test_idx)}")

dict_keys(['cnv', 'miRNA', 'rna', 'methylation'])
Shared N=8034 | train=6409 | val=798| test=800


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

In [5]:
train_loader, val_loader, test_loader = make_loaders_from_splits(
    multi_omic_data, common_samples, train_idx, val_idx, test_idx, batch_size=64
)

In [6]:
# 1) Rebuild encoders/decoders/hidden_dims and mask_values
encoders, decoders, hidden_dims = {}, {}, {}
mask_values = {}

for mod in multi_omic_data.keys():
    ae_m, hidden_dim_m, cfg_m = load_modality_with_config(
        f"aes_redo_z/{mod}_ae.pt", map_location=device
    )
    ae_m = ae_m.to(device)

    enc, dec = extract_encoder_decoder_from_pretrained(ae_m)
    encoders[mod] = enc
    decoders[mod] = dec
    hidden_dims[mod] = hidden_dim_m

    # modality-specific sentinel (default 0.0 if missing)
    mask_values[mod] = cfg_m.get("mask_value", 0.0)

print("mask_values:", mask_values)

# 2) Load checkpoint into shared-space model
checkpoint_path = "checkpoints/fintuned/shared_model_ep200.pt"
model = load_shared_model(
    MultiModalWithSharedSpace,
    encoders,
    decoders,
    hidden_dims,
    shared_dim=256,
    proj_depth=1,
    checkpoint_path=checkpoint_path,
    map_location=device
)
model = model.to(device)
model.eval()


/net/noble/vol1/home/nambiar4/Documents/Final/src/mae_masked.py:742: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path, map_location=map_location)


mask_values: {'cnv': 0.0, 'miRNA': 0, 'rna': 0, 'methylation': 0.0}


/net/noble/vol1/home/nambiar4/Documents/Final/src/shared_finetune.py:204: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map

[Loaded] Shared model from checkpoints/fintuned/shared_model_ep200.pt


MultiModalWithSharedSpace(
  (encoders): ModuleDict(
    (cnv): ModalityEncoder(
      (net): Sequential(
        (0): Linear(in_features=3105, out_features=256, bias=True)
      )
    )
    (miRNA): ModalityEncoder(
      (net): Sequential(
        (0): Linear(in_features=383, out_features=128, bias=True)
      )
    )
    (rna): ModalityEncoder(
      (net): Sequential(
        (0): Linear(in_features=3007, out_features=512, bias=True)
      )
    )
    (methylation): ModalityEncoder(
      (net): Sequential(
        (0): Linear(in_features=3139, out_features=256, bias=True)
      )
    )
  )
  (decoders): ModuleDict(
    (cnv): ModalityDecoder(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=3105, bias=True)
      )
    )
    (miRNA): ModalityDecoder(
      (net): Sequential(
        (0): Linear(in_features=128, out_features=383, bias=True)
      )
    )
    (rna): ModalityDecoder(
      (net): Sequential(
        (0): Linear(in_features=512, out_features=

## 0.2 Missing Completely At Random (MCAR) imputation

We first generate MCAR missingness on the **validation set**.
These results are used exclusively for tuning baseline methods (e.g. SoftImpute),
and are not used for final evaluation.


In [7]:
multi_ds = MultiOmicDataset({m: df for m, df in multi_omic_data.items()})
val_samples = [multi_ds.common_samples[i] for i in val_idx]
mask_dfs, pred_dfs = mask_and_predict(
    model, mask_values, multi_omic_data, val_samples,
    use_modalities=["rna", "miRNA", "methylation", "cnv"],
    mask_modalities=["rna", "miRNA", "methylation", "cnv"],
    masking_policy="random",
    masking_fraction=0.3,
    seed=1,
    batch_size=256,
    device=device,
    save_mask_pickle_path="missing_values_data/val_MCAR30_masks.pkl",
    save_pred_pickle_path="missing_values_data/val_MCAR30_preds.pkl",
    save_corrupt_pickle_path="missing_values_data/val_MCAR30_corrupt.pkl",
    self_weight = 10.0,
)

[Saved mask dict] val_MCAR30_masks.pkl
[Saved predictions dict] val_MCAR30_preds.pkl
[Saved corrupted data dict] val_MCAR30_corrupt.pkl


For each modality, a fixed fraction (30%) of values is randomly masked.
The model imputes these values using the remaining observed entries.
Masks, corrupted inputs, and predictions are saved to disk for reuse.

### 0.2.2 MCAR imputation on the test set

Using the same MCAR masking parameters, we generate missing-value imputations
on the held-out test set.


In [8]:
multi_ds = MultiOmicDataset({m: df for m, df in multi_omic_data.items()})
test_samples = [multi_ds.common_samples[i] for i in test_idx]
mask_dfs, pred_dfs = mask_and_predict(
    model, mask_values, multi_omic_data, test_samples,
    use_modalities=["rna", "miRNA", "methylation", "cnv"],
    mask_modalities=["rna", "miRNA", "methylation", "cnv"],
    masking_policy="random",
    masking_fraction=0.3,
    seed=1,
    batch_size=256,
    device=device,
    save_mask_pickle_path="missing_values_data/test_MCAR30_masks.pkl",
    save_pred_pickle_path="missing_values_data/test_MCAR30_preds.pkl",
    save_corrupt_pickle_path="missing_values_data/test_MCAR30_corrupt.pkl",
    self_weight = 10.0,
)

[Saved mask dict] test_MCAR30_masks.pkl
[Saved predictions dict] test_MCAR30_preds.pkl
[Saved corrupted data dict] test_MCAR30_corrupt.pkl


## 0.3 Missing Not At Random (MNAR) imputation

We next generate MNAR missingness on the test set.
In this setting, missingness depends on the value itself:
lower-valued entries are more likely to be masked.


In [9]:
multi_ds = MultiOmicDataset({m: df for m, df in multi_omic_data.items()})
test_samples = [multi_ds.common_samples[i] for i in test_idx]
mask_dfs, pred_dfs = mask_and_predict(
    model, mask_values, multi_omic_data, test_samples,
    use_modalities=["rna", "miRNA", "methylation", "cnv"],
    mask_modalities=["rna", "miRNA"],
    masking_policy="low_vals",
    masking_fraction=0.3,
    low_vals_alpha=2.0,
    low_vals_transform="rank",
    seed=1,
    batch_size=256,
    device=device,
    save_mask_pickle_path="missing_values_data/test_MNAR30_masks.pkl",
    save_pred_pickle_path="missing_values_data/test_MNAR30_preds.pkl",
    save_corrupt_pickle_path="missing_values_data/test_MNAR30_corrupt.pkl",
    self_weight = 10.0,
)


[Saved mask dict] test_MNAR30_masks.pkl
[Saved predictions dict] test_MNAR30_preds.pkl
[Saved corrupted data dict] test_MNAR30_corrupt.pkl


MNAR configuration:
- Masked modalities: mRNA and miRNA only
- Masking fraction: 30%
- Masking policy: value-dependent ("low_vals")
- Ranking transform: rank-based
- Alpha (strength): 2.0
